In [ ]:
import time, math
import numpy as np
import PIL.Image
from pynq import Overlay, MMIO, allocate
import svo_builder

BITSTREAM = '/home/xilinx/jupyter_notebooks/svo_system.bit'
IMG_W, IMG_H = 320, 240
BYTES_PER_PIXEL = 4   # 32-bit XRGB

def to_q16(f): return int(f * 65536) & 0xFFFF_FFFF
def pack_rgb(r, g, b): return (int(r)&0xFF)|((int(g)&0xFF)<<8)|((int(b)&0xFF)<<16)
def normalise(v):
    l = math.sqrt(sum(x**2 for x in v))
    return [x/l for x in v] if l > 1e-9 else v
def cross(a, b):
    return [a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0]]

# ---------------------------------------------------------------------------
# Load bitstream
# ---------------------------------------------------------------------------
print('Loading bitstream ...')
ol = Overlay(BITSTREAM)
ip = ol.top_0

# ---------------------------------------------------------------------------
# Configure VDMA S2MM via direct MMIO (AXI VDMA v6.3, PG020)
# S2MM register offsets (verified from HWH JSON — NOTE: different from MM2S):
#   0x30  S2MM_VDMACR       control (bit0=RS, bit1=Circular_Park, bit2=Reset)
#   0x34  S2MM_VDMASR       status  (bit0=Halted, bit4=IntErr, bit5=SlvErr, bit6=DecErr)
#   0xA0  S2MM_VSIZE        lines   (write LAST — arms channel, asserts tready)
#   0xA4  S2MM_HSIZE        bytes per line
#   0xA8  S2MM_FRMDLY_STRIDE
#   0xAC/0xB0/0xB4  S2MM_SA1/2/3  (all 3 required; C_NUM_FSTORES=3)
# ---------------------------------------------------------------------------
VDMA_BASE = ol.ip_dict['axi_vdma_0']['phys_addr']
vdma = MMIO(VDMA_BASE, 0x1000)

HSIZE  = IMG_W * BYTES_PER_PIXEL
STRIDE = IMG_W * BYTES_PER_PIXEL

frame_buf  = allocate(shape=(IMG_H, IMG_W, BYTES_PER_PIXEL), dtype=np.uint8)
frame_phys = frame_buf.physical_address

# Reset S2MM and wait for self-clear
vdma.write(0x30, 0x4)
while vdma.read(0x30) & 0x4: pass

# All 3 frame buffer start addresses (same physical buffer for all 3 stores)
vdma.write(0xAC, frame_phys)
vdma.write(0xB0, frame_phys)
vdma.write(0xB4, frame_phys)

vdma.write(0xA8, STRIDE)
vdma.write(0xA4, HSIZE)
vdma.write(0x30, 0x3)       # RS=1, Circular_Park=1
vdma.write(0xA0, IMG_H)     # VSIZE last — arms channel, tready asserts

s2mm_sr = vdma.read(0x34)
print(f'VDMA S2MM ready: phys=0x{frame_phys:08X}, status=0x{s2mm_sr:08X}')
if s2mm_sr & 0x1:
    print('WARNING: S2MM is Halted — check HSIZE/VSIZE/addresses and re-run')

# ---------------------------------------------------------------------------
# Build and upload SVO
# ---------------------------------------------------------------------------
print('Building SVO ...')
grid  = svo_builder.build_world()
root  = svo_builder.build_svo(grid)
nodes = svo_builder.flatten_svo(root)
words = svo_builder.serialise_nodes(nodes)
print(f'  {len(nodes)} nodes -> {len(words)} words')
ip.write(0x48, 0)
for w in words: ip.write(0x4C, w)

# Colour registers
for offset, rgb in zip(range(0x50, 0x70, 4), [
    (0,0,0),(128,128,128),(60,160,40),(255,220,80),(0,0,0),(0,0,0),(135,206,235),(180,200,220)]):
    ip.write(offset, pack_rgb(*rgb))
ip.write(0x70, to_q16(15.0))
ip.write(0x74, to_q16(0.01))

# Camera
pos   = [32.0, 40.0, -20.0]
fwd   = normalise([32.0-pos[0], 4.0-pos[1], 32.0-pos[2]])
right = normalise(cross(fwd, [0,1,0]))
up    = cross(right, fwd)
fov_scale = math.tan(math.radians(60)/2) / (IMG_W/2)
ip.write(0x38, to_q16(fov_scale))
for offset, val in zip(range(0x08, 0x38, 4), pos+right+up+fwd):
    ip.write(offset, to_q16(val))
ld = normalise([0.5,-0.7,0.5])
ip.write(0x3C, to_q16(ld[0])); ip.write(0x40, to_q16(ld[1])); ip.write(0x44, to_q16(ld[2]))

# ---------------------------------------------------------------------------
# Trigger render and poll for completion
# ---------------------------------------------------------------------------
print('Triggering render ...')
t0 = time.time()
ip.write(0x00, 1)
while ip.read(0x04) & 0x1: time.sleep(0.001)
elapsed = time.time() - t0
print(f'Done in {elapsed:.3f} s  ({1/elapsed:.2f} FPS)')

# S2MM has no Idle bit — give the line buffer time to flush to DDR
time.sleep(0.05)
s2mm_sr = vdma.read(0x34)
if s2mm_sr & 0x70:   # IntErr | SlvErr | DecErr
    print(f'WARNING: VDMA error bits set: status=0x{s2mm_sr:08X}')

# Invalidate cache and read frame ([B,G,R,pad] -> RGB)
frame_buf.invalidate()
frame_rgb = np.array(frame_buf[:, :, [2, 1, 0]])
image = PIL.Image.fromarray(frame_rgb, 'RGB')

In [ ]:
# Display inline
image

In [ ]:
# Stop VDMA and free frame buffer
vdma.write(0x30, 0x0)   # RS=0 (stop)
frame_buf.freebuffer()